# Mini LLM Inference Server — Project Overview

Welcome! This project guides you through building a **production-grade LLM inference server from scratch**, one function at a time.

You will implement **58 problems** across **7 parts**, each building on the last, until you have a fully working server that can serve language model generations over HTTP with streaming support.

## System Architecture

```
┌─────────────────────────────────────────────────────────────────┐
│                   Mini LLM Inference Server                     │
│                                                                 │
│  HTTP Client                                                    │
│      │                                                          │
│      ▼                                                          │
│  ┌──────────────────────────────────────────┐                  │
│  │           FastAPI Layer (Part 6)          │                  │
│  │  POST /generate   POST /generate/stream   │                  │
│  └────────────────────┬─────────────────────┘                  │
│                       │                                         │
│                       ▼                                         │
│  ┌──────────────────────────────────────────┐                  │
│  │    Continuous Batching Engine (Part 5)    │                  │
│  │  Request Queue → Scheduler → Batch Loop   │                  │
│  └────────────────────┬─────────────────────┘                  │
│                       │                                         │
│           ┌───────────┴───────────┐                            │
│           ▼                       ▼                            │
│  ┌─────────────────┐   ┌──────────────────────┐               │
│  │ Paged Attention  │   │  KV Cache (Part 3)   │               │
│  │    (Part 4)      │   │  Prefill + Decode    │               │
│  │  Page Pool +     │   │  Buffer allocation   │               │
│  │  Block Table     │   │  Read/Write ops      │               │
│  └────────┬─────────┘   └──────────┬───────────┘               │
│           └───────────┬────────────┘                           │
│                       ▼                                         │
│  ┌──────────────────────────────────────────┐                  │
│  │       Transformer Model (Part 1)          │                  │
│  │  Embed → Attention → FFN → Logits         │                  │
│  └────────────────────┬─────────────────────┘                  │
│                       │                                         │
│                       ▼                                         │
│  ┌──────────────────────────────────────────┐                  │
│  │         Sampling (Part 2)                 │                  │
│  │  Greedy / Temperature / Top-K / Top-P     │                  │
│  └────────────────────┬─────────────────────┘                  │
│                       │                                         │
│                       ▼                                         │
│              Next Token ID                                      │
│                                                                 │
│  ┌──────────────────────────────────────────┐                  │
│  │         Benchmarks (Part 7)               │                  │
│  │  TTFT · TPOT · Throughput · Reports       │                  │
│  └──────────────────────────────────────────┘                  │
└─────────────────────────────────────────────────────────────────┘
```

## The 7 Parts

| Part | Problems | What You Build |
|------|----------|----------------|
| 1 | 001–009 | Tiny Transformer: vocab, attention, transformer blocks, GPT forward pass |
| 2 | 010–016 | Sampling: greedy, temperature, top-k, top-p, autoregressive generation |
| 3 | 017–024 | KV Cache: allocate buffers, prefill, decode, benchmark speedup |
| 4 | 025–033 | Paged Attention: page pool, block table, fragmentation-free KV storage |
| 5 | 034–042 | Continuous Batching: request queue, scheduler, iteration-level batching |
| 6 | 043–050 | Streaming API: FastAPI server, SSE streaming, request tracking |
| 7 | 051–058 | Benchmarks: TTFT, TPOT, latency plots, throughput plots, reports |

## How to Use This Project

### Workflow for each problem

```bash
# 1. Read the problem stub
cat problems/001_build_token_vocab.py

# 2. Copy it to solutions/
cp problems/001_build_token_vocab.py solutions/001_build_token_vocab.py

# 3. Open the solution file and implement the function
# (replace 'raise NotImplementedError' with your code)

# 4. Run the tests
pytest tests/test_001_build_token_vocab.py -v

# 5. Submit when all tests pass
python3 submit.py 001
```

### Running the server

```bash
# After solving problems 045, 047, 048:
python3 main.py

# Test it
curl -X POST http://localhost:8000/generate \
  -H 'Content-Type: application/json' \
  -d '{"prompt": "Hello world", "max_tokens": 20}'
```

## Full Data Flow: Prompt → Token

Here is the complete journey of a single generation request:

```
User sends:  "The quick brown"
                    │
                    ▼
           [Tokenisation - Part 1]
           "The" → 464
           "quick" → 2068
           "brown" → 7586
           token_ids = [464, 2068, 7586]
                    │
                    ▼
           [Embedding lookup]
           token_ids → tensor shape [3, d_model]
           + positional encodings
                    │
                    ▼
           [Transformer Blocks × N - Part 1]
           Each block: LayerNorm → Attention → LayerNorm → FFN
           hidden states shape: [3, d_model]
                    │
                    ▼
           [Language Model Head]
           hidden[-1] → logits shape [vocab_size]  (50257 for GPT-2)
                    │
                    ▼
           [Sampling - Part 2]
           temperature=0.8, top_k=50, top_p=0.95
           logits → probabilities → sample → token_id = 312
                    │
                    ▼
           [Decode - Part 1]
           312 → "fox"
                    │
                    ▼
           Response: "The quick brown fox"
                    │  (repeat for next token)
                    ▼
                  ...
```

In [ ]:
import sys
from pathlib import Path

# Make sure the project root is on sys.path so solutions/ is importable
project_root = Path('__file__').parent.parent if '__file__' in dir() else Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
# Also try the current directory's parent
for p in [Path.cwd().parent, Path.cwd().parent.parent]:
    if (p / 'solutions').exists() and str(p) not in sys.path:
        sys.path.insert(0, str(p))
        break


In [ ]:
# Quick project status check
import pathlib
import importlib.util

project_root = pathlib.Path("..")
solutions_dir = project_root / "solutions"
problems_dir = project_root / "problems"

problem_files = sorted(f for f in problems_dir.glob("[0-9]*.py"))
solution_files = {f.name for f in solutions_dir.glob("[0-9]*.py")}

solved = [f.name in solution_files for f in problem_files]
total = len(problem_files)
done = sum(solved)

print(f"Progress: {done}/{total} problems solved")
print()

parts = [
    ("Part 1 – Tiny Transformer  ", range(1, 10)),
    ("Part 2 – Sampling          ", range(10, 17)),
    ("Part 3 – KV Cache          ", range(17, 25)),
    ("Part 4 – Paged Attention   ", range(25, 34)),
    ("Part 5 – Continuous Batching", range(34, 43)),
    ("Part 6 – Streaming API     ", range(43, 51)),
    ("Part 7 – Benchmarks        ", range(51, 59)),
]

for part_name, nums in parts:
    part_files = [f"{n:03d}" for n in nums]
    part_solved = sum(1 for pf in part_files
                      if any(sf.startswith(pf) for sf in solution_files))
    part_total = len(part_files)
    bar = "#" * part_solved + "." * (part_total - part_solved)
    pct = 100 * part_solved // part_total if part_total else 0
    print(f"  {part_name}  [{bar}]  {part_solved}/{part_total}  ({pct}%)")

## Key Concepts You Will Learn

- **Tokenisation**: How raw text becomes integers a model can process
- **Transformer Architecture**: Self-attention, multi-head attention, feed-forward blocks
- **Autoregressive Generation**: How one token is produced at a time
- **KV Caching**: How to avoid recomputing attention keys/values for previous tokens
- **Paged Attention**: vLLM's memory-efficient approach using fixed-size pages
- **Continuous Batching**: How to serve many users efficiently without wasted GPU steps
- **SSE Streaming**: How to stream tokens back to the client as they are generated
- **Benchmarking**: How to measure TTFT, TPOT, throughput, and latency

Good luck! Start with `notebooks/01_tiny_transformer.ipynb`.